### Importing Libraries

In [1]:
%pip install rapidfuzz
import pandas as pd
import io
import os
import re
import rapidfuzz
import joblib
import numpy as np
from rapidfuzz import fuzz, process
from scipy.spatial import distance
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import FeatureUnion, Pipeline
from datetime import datetime, date, timedelta
import pandas as pd
from io import BytesIO
from ipywidgets import FileUpload

def select_excel_file():
    uploader = FileUpload(accept='.xlsx,.xls,.xlsm', multiple=False)
    uploader
    return uploader

Note: you may need to restart the kernel to use updated packages.


### Select Report

In [2]:
excelfile = select_excel_file()
excelfile

FileUpload(value=(), accept='.xlsx,.xls,.xlsm', description='Upload')

### Defining Functions

In [4]:
if not excelfile.value:
    raise FileNotFoundError("No file uploaded")
val = excelfile.value
if isinstance(val, dict):
    file_data = next(iter(val.values()))
elif isinstance(val, tuple):
    file_data = val[0]
else:
    raise TypeError("Unknown FileUpload.value format")
file_bytes = file_data["content"]

def read_excel_bytes(file_bytes):
    # required_sheets = ["YgnTBPro", "Presumptive Register", "Notified TB Register", "Outcome", "TPT Register"]
    required_sheets = ["YgnTBPro", "Presumptive Register", "Notified TB Register","TPT Register"]

    with pd.ExcelFile(BytesIO(file_bytes), engine="openpyxl") as xf:
        try:
            dfs = pd.read_excel(
                        BytesIO(file_bytes),
                        sheet_name=required_sheets)
            for name in required_sheets:
               dfs[name]['SheetName'] = name
            return dfs
        except Exception:
            print("Available sheets:", xf.sheet_names)
            raise
df_dict = read_excel_bytes(file_bytes)


In [5]:
def clean_column_name(col_name):
  col_name = re.sub(r'[\\/,()\n]|_x000D_|\b(BC/CD|DS_TB/DR_TB)\b', '', col_name).strip()
  rename_dict = {
            'Type of TB Treatment DS_TBDR_TB': 'Type of TB Treatment',
            'Treatment Registration YN': 'Treatment Registration',
            'Treatment Registrar Dateplease fill in date format: 1-Jan-2021': 'Treatment Registration Date',
            'NationalityN  NN':'Nationality',
            'Specify ______ if EPSite of EPTB':'Specify if EPTB',
            'CPT please fill in date format: 1-Jan-2021':'CPT Date',
            'ARTplease fill in date format: 1-Jan-2021':'ART Date',
            'Xpert Result N= MTB not detected I = Invalid No result  T= MTB detected RR= Rif resistant TI= MTB +  Rif resistant is invalid or No result':'Gene Xpert Result',
            'Initial Regimen Started Dateplease fill in date format: 1-Jan-2021':'Initial Regime Started Date',
            "Type of Patient's":"Type Of Patient"
        }
  return rename_dict.get(col_name, col_name)

In [6]:
def changeDtype(df):
  missingcolumns =[]
  dtype_change = {
            'Date_x': 'datetime64[ns]', 'Team': 'Int64', 'Tsp': 'string', 'WardVillage': 'string',
            'WardCode': 'string', 'Approach': 'string', 'Visit_no': 'Int64', 'Sr_No': 'Int64',
            'PatientID': 'string', 'Name_x': 'string', 'Age_x': 'float32', 'Sex_x': 'Int64',
            'Address_x': 'string', 'Ptstsp_x': 'string', 'PhoneNo_x': 'string', 'Origin': 'Int64',
            'VOL': 'Int64', 'Referralfor': 'Int64', 'VolunteerName': 'string', 'Organization': 'string',
            'Cough': 'Int64', 'Fever': 'Int64', 'Wtloss': 'Int64', 'Nightsweat': 'Int64', 'Haemoptysis': 'Int64',
            'Chestpain': 'Int64', 'Fatigue': 'Int64', 'Neckglands': 'Int64', 'TBcontact': 'Int64',
            'MDRTBcontact': 'Int64', 'TBTreatmenthistory': 'Int64',
            'Smoking': 'Int64', 'Reasonforexamination': 'Int64',
            'TypeofPatient': 'Int64', 'PublicHealthCare1': 'Int64', 'TypeofPatient1': 'Int64',
            'DM1': 'Int64', 'HT1': 'Int64','RBS':'Int64', 'HbA1c':'Int64', 'Systolic':'Int64','Diastolic':'Int64',
            'DMHT1': 'Int64', 'RTIAVI1': 'Int64', 'Generalweakness1': 'Int64',
            'Other1': 'string', 'Cxrr': 'Int64', 'CXRresult': 'Int64', 'Cxr_others': 'string',
            'Sputum_request': 'Int64', 'Micror': 'Int64', 'Sputummicroscopyresult': 'Int64',
            'Genexpertrequested': 'Int64', 'GeneXpertresult': 'Int64', 'XDRResult':'string','CultureDSTresult': 'string',
            'Bact_status': 'Int64', 'Case': 'Int64', 'Treatmentreferral': 'Int64','OLDTYPEOFTBTREATMENT':'string', 'TreatmentRegimen': 'Int64',
            'Modifiedregimen':'string','Placeforreferral': 'Int64', 'Ref1': 'string', 'TreatmentOutcome1211': 'Int64','Tx_Outcome_Date':'datetime64[ns]',
            'DOTvolName111':'string','VolunteerGender111':'string','VolunteerOrganization111':'string','ContactInvestigation111':'string',
            'DOTSupervision111':'string','DOTStartedDate111':'string','DOTsupervisiontillTreatmentComp111':'string',
            'Seeing1': 'Int64', 'Hearing1': 'Int64', 'Walking1': 'Int64', 'Cognition1': 'Int64',
            'Selfcare1': 'Int64', 'Communication1': 'Int64', 'Disability1': 'Int64', 'Remark_x': 'string',
            'MonthDiagnosis11': 'string', 'Location11': 'string', 'EPI11': 'string',
            'TB11': 'string', 'CXRresult211': 'Int64', 'Xray2ndReading11': 'Int64',
            'XrayRemark11': 'string', 'XrayEQA11': 'string', 'TownshipTBRegNumber': 'string',
            'ReportingTsp11': 'string', 'ReportingYear11': 'Int64', 'Recstatus': 'string',
            'UniqueKey': 'string', 'Month Diagnosis': 'string', 'Place of Notification': 'string',
            'Name_y': 'string', 'Address_y': 'string', 'Ptstsp_y': 'string', 'Ward': 'string',
            'PhoneNo_y': 'string', 'Volunteer Referral': 'string', 'Bacteriological status': 'string',
            'Type of TB Treatment': 'string', 'Treatment Registration': 'string',
            'StateRegionName': 'string', 'StateRegion Name':'string','Township Name': 'string', 
            'Facility Name Place of Registration': 'string','ReportingPeriod': 'string', 'RegistrarDate': 'datetime64[ns]',
            'Reporting Period':'string','Treatment Registrar Date':'datetime64[ns]','Township TB Reg Number':'string',
            'TownshipTBRegNumber': 'string', 'Nationality_x': 'string', 'Nationality_y': 'string', 'Sex_y': 'string',
            'Age_y': 'float32', 'Refered from': 'string', 'Transferin': 'string', 'Transfer in':'string','Type of Patient':'string',
            'TB Site':'string','Specify Site of EPTB if EP':'string','Treatment Regimens':'string',
            "TypeofPatients": 'string', 'TypeofDisease': 'string', 'SpecifyifEPSiteofEPTB': 'string',
            'Treatment Regimens': 'string', 'Smoking Status': 'string', 'DM Status': 'string',
            'HIVStatus': 'string', 'CPT': 'datetime64[ns]', 'ART': 'datetime64[ns]', 'HIV Status':'string',
            'CPT Date':'datetime64[ns]','ART Date':'datetime64[ns]',
            'Microscope Result': 'string', 'X-Ray Result': 'Int64', 'Xpert Result': 'string',
            'Culture Result': 'string', '2nd month Microscope Result': 'string',
            '2nd month Xpert Result': 'string', '3rd month Microscope Result': 'string',
            '3rd month Xpert Result': 'string', '5th month Microscope Result': 'string',
            '5th month Xpert Result': 'string', 'End of Tx Microscope Result': 'string',
            'End of Tx X-Ray Result': 'string', 'End of Tx Xpert Result': 'string','TreatmentOutcome1211':'Int64',
            'Treatment Outcome': 'string', 'Initial Regimen Started Date': 'datetime64[ns]',
            'Treatment Outcome date': 'datetime64[ns]', 'Volunteer Name': 'string',
            'Volunteer Gender': 'string', 'Volunteer Organization': 'string',
            'Contact Investigation': 'string','DOT  Supervision': 'string',
            'DOT Started Date': 'datetime64[ns]',
            'DOT supervision till Treatment Completion': 'string', 'Seeing': 'Int64',
            'Hearing': 'Int64', 'Walking': 'Int64', 'Cognition': 'Int64', 'Selfcare': 'Int64',
            'Communication': 'Int64', 'Disability': 'Int64', 'Remark_y': 'string',
            'SDG_EPI': 'string','SDG_TB': 'string', 'Location': 'string', 'Date_y': 'datetime64[ns]', 'Reporting Year': 'Int64',
            'SheetName_y': 'string', 'TPTregno':'string','RegnoofindexTBcase':'string','TPTregistrationdate':'datetime64[ns]',
            'TPTregimen':'string','DrugDose':'string','Drugpickup1':'datetime64[ns]','Date_drug_pickup2':'datetime64[ns]',
            'Date_drug_pickup_3':'datetime64[ns]','Date_drug_pickup_4':'datetime64[ns]','Date_drug_pickup_5':'datetime64[ns]',
            'Date_drug_pickup_6':'datetime64[ns]','Date_drug_pickup_7':'datetime64[ns]','Date_drug_pickup_8':'datetime64[ns]',
            'Date_drug_pickup_9':'datetime64[ns]','TPTdiscontinuationdate':'datetime64[ns]','TPTTreatmentOutcome':'string',
            'OutcomeDate':'datetime64[ns]','TypeofTBTreatment':'Int64','SheetName_x':'string'
        }
  
        # Convert each column based on dtype_change
  for col, dtype in dtype_change.items():
    if col in df.columns: # Check if the column exists before attempting to convert
      if dtype.startswith('datetime64'):
        df[col] = pd.to_datetime(df[col].replace("", np.nan), errors='coerce')
      elif dtype == 'Int64':  # Nullable integer type (allows NaN)
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
      elif dtype == 'float32':
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')
      elif dtype == 'string':
        df[col] = df[col].astype('string')
    else:
      #print(f"Column '{col}' not found in DataFrame. Skipping dtype conversion.")
      missingcolumns.append(col)
  print(missingcolumns)
  return df

In [7]:

class TBDataValidator:
    def __init__(self):
        self.invalid_rows = []
        self.visit_log = {}
        self.target_year = 2026
        self.existing_tb_regs = set()

        self.mappings = {
            'sex': {1: "M", 2: "F"},
            'yes_no': {1: "Yes", 2: "No"},
            'y_n': {1: "Y", 2: "N"},
            'tx_regime': {1: "I", 2: "R", 3: "C", 4: "MDR", 5: "M"},
            'bact_status': {1: "BC", 2: "CD"},
            'facility': {1: "1. NTP", 2: "2. MMA-PPM", 3: "3. PSI-PPM", 4: "4. MATA-PPM", 5: "5. Other"},
            'smoking': {0: "N", 1: "C", 2: "P"},
            'dm': {0: "U", 1: "Y",2: "N",3:"Y"},
            'dsdr': {1: "DS-TB", 2: "DR-TB"},
            'gene': {0: "N", 1: "I", 2: "T", 3: "RR", 4: "TI", 5: "denied", 6: "missing", 7: "TT"}
        }
        self.date_columns_to_check = [
                    'Date_y','Date_x','RegistrarDate','CPT', 'ART','Tx_Outcome_Date',
                    'DOTvolName111','DOTStartedDate111','TPTregistrationdate','Drugpickup1', 
                    'Date_drug_pickup2', 'Date_drug_pickup_3', 'Date_drug_pickup_4', 
                    'Date_drug_pickup_5', 'Date_drug_pickup_6', 'Date_drug_pickup_7', 
                    'Date_drug_pickup_8', 'Date_drug_pickup_9', 'TPTdiscontinuationdate', 
                    'OutcomeDate', 'Treatment Registrar Date', 'CPT Date', 'ART Date', 
                    'Initial Regimen Started Date', 'Treatment Outcome date','DOT Started Date'
                ]
        
    def is_missing(self, data):
        if pd.isna(data):
            return True
        if str(data).strip().lower() in ["", "nan", "none"]:
            return True
        return False

    def is_in(self, val, options):
        if pd.isna(val): 
            return False
        return bool(val in options)
    
    def is_equal(self, val, target):
        if pd.isna(val):
            return False
        return bool(val == target)
    
    def add_error(self, row, message):
        self.invalid_rows.append({
            "InvalidData": message,
            "PatientID": row.get('PatientID', ''),
            "Approach": row.get('Approach', ''),
            "Tsp": row.get('Tsp', '')
        })

    def check_year_consistency(self, row):
        for col in self.date_columns_to_check:
            date_val = row.get(col)
            if not self.is_missing(date_val):
                dt_obj = pd.to_datetime(date_val, errors='coerce')
                if not pd.isna(dt_obj) and dt_obj.year != self.target_year:
                    self.add_error(row, f"checkYear in {col}: Year is {dt_obj.strftime('%d-%b-%Y')}") # {dt_obj.year}

    def check_date(self, row):
        date_field = row.get('Date_x')
        visit_no = row.get('Visit_no')
        approach = row.get('Approach')

        if self.is_missing(date_field) or self.is_missing(visit_no) or self.is_missing(approach):
            return

        visit_date = pd.to_datetime(date_field, errors='coerce')
        if pd.isna(visit_date): 
            return

        if self.is_in(approach, ["PPM", "Diagnostic Center"]):
            try:
                visit_month = int(visit_no)
                if 1 <= visit_month <= 12:
                    curr_year = visit_date.year 
                    if visit_month == 1:
                        start_date = pd.Timestamp(year=curr_year - 1, month=12, day=26)
                        end_date = pd.Timestamp(year=curr_year, month=1, day=25)
                    else:
                        start_date = pd.Timestamp(year=curr_year, month=visit_month - 1, day=26)
                        end_date = pd.Timestamp(year=curr_year, month=visit_month, day=25)
                    if not (start_date <= visit_date <= end_date):
                        self.add_error(row, f"Date {visit_date.date()} outside PPM range ({start_date.date()} to {end_date.date()})")
                else:
                    self.add_error(row, f"Invalid Month: {visit_no}")
            except (ValueError, TypeError):
                pass
        else:
            if visit_no in self.visit_log:
                last_date = pd.to_datetime(self.visit_log[visit_no])
                date_diff = abs((visit_date - last_date).days)
                
                if date_diff > 30:
                    self.add_error(row, f"Visit {visit_no} discrepancy: {date_diff} days difference")
            else:
                self.visit_log[visit_no] = visit_date

    def check_patientid(self, row):
            pid = str(row.get('PatientID', '')).strip()
            if self.is_missing(pid) or len(pid) < 12:
                self.add_error(row, f"check_PatientID: Invalid PatientID {pid}")
                return
            
            # --- Duplicate Check ---
            if not hasattr(self, 'seen_pids'):
                self.seen_pids = set()
                
            if pid in self.seen_pids:
                self.add_error(row, f"check_PatientID: Duplicate PatientID detected: {pid}")
            else:
                self.seen_pids.add(pid)
            # -----------------------

            if (self.is_missing(row.get('Visit_no')) or self.is_missing(row.get('Date_x')) or 
                self.is_missing(row.get('Sr_No')) or self.is_missing(row.get('Team')) or 
                self.is_missing(row.get('Tsp')) or self.is_missing(row.get('Approach'))):
                
                self.add_error(row, f"missing: Year={pd.to_datetime(row.get('Date_x'), errors='coerce').date()}, VisitNo={row.get('Visit_no')}, SerialNo={row.get('Sr_No')}, Team={row.get('Team')}, Approach={row.get('Approach')}, Tsp={row.get('Tsp')}")

            team = str(row.get('Team', '')).strip()
            tsp = str(row.get('Tsp', '')).strip()
            app = str(row.get('Approach', '')).strip()
            
            raw_vn = row.get('Visit_no')
            vn = str(int(float(raw_vn))).zfill(2) if not self.is_missing(raw_vn) else ''
            
            raw_sr = row.get('Sr_No')
            sr = str(int(float(raw_sr))).zfill(4) if not self.is_missing(raw_sr) else ''
            
            # Safe Date Guard evaluation order
            raw_date = row.get('Date_x')
            yr = raw_date.strftime('%y') if (not self.is_missing(raw_date) and hasattr(raw_date, 'strftime')) else ''

            if self.is_equal(team, '5') and team != pid[0]:
                self.add_error(row, f"check_PatientID: Organization {pid[0]} mismatch with Team {team}")
            
            if self.is_in(tsp, ["SDG", "KMD", "HLG", "HTY"]):
                if tsp != pid[1:4]:
                    self.add_error(row, f"check_PatientID: Tsp {pid[1:4]} mismatch with {tsp}")

            app_map = {"PPM": "P", "Diagnostic Center": "D", "Mobile Visit": "M", "Elderly Care": "E", "Touring": "T"}
            if self.is_in(app, app_map.keys()):
                app_char = app_map[app]
                if pid[8] != app_char:
                    self.add_error(row, f"check_PatientID: Approach {pid[8]} mismatch with {app}")

            # Use the raw metrics here to prevent matching against broken '0nan' strings
            if not self.is_missing(raw_vn):
                if vn != pid[6:8]:
                    self.add_error(row, f"check_PatientID: Visit {pid[6:8]} vs {vn}")
                    
            if not self.is_missing(raw_date):
                if yr != pid[4:6]:
                    self.add_error(row, f"check_PatientID: Year {pid[4:6]} vs {yr}")
                    
            if not self.is_missing(raw_sr) and pid.endswith('S'):
                if sr != pid[9:]:
                    self.add_error(row, f"check_PatientID: Serial {pid[9:]} vs {sr}")

    def check_CXR_GXP(self, row):
        cxr_req, cxr_res = row.get('Cxrr'), row.get('CXRresult')
        gxp_req, gxp_res = row.get('Genexpertrequested'), row.get('GeneXpertresult')
        sputum_req = row.get('Sputum_request')
        micro_req, micro_res = row.get('Micror'), row.get('Sputummicroscopyresult')

        if self.is_equal(cxr_req, 1) and (self.is_missing(cxr_res) or not (1 <= cxr_res <= 5)):
            if 'still on tx' not in str(row.get('Remark_x', '')).lower():
                self.add_error(row, "check_CXR: missing CXR result although requested")

        if self.is_equal(sputum_req, 1) and self.is_missing(micro_req) and self.is_missing(gxp_req):
            self.add_error(row, "check_sputum: Sputum requested but no Microscope or GXP request")

        if self.is_equal(gxp_req, 1) and (self.is_missing(gxp_res) or not (0 <= gxp_res <= 7)):
            if 'waiting result' not in str(row.get('Remark_x', '')).lower():
                self.add_error(row, "check_GeneXpert: missing GXP result although requested")

        if self.is_equal(micro_req, 1) and (self.is_missing(micro_res) or not (0 <= micro_res <= 7)):
            if 'waiting result' not in str(row.get('Remark_x', '')).lower():
                self.add_error(row, "check_Microscope: missing Microscope result although requested")

    def check_bact_status(self, row):
        bact = row.get('Bact_status')
        sputum = row.get('Sputummicroscopyresult')
        gene = row.get('GeneXpertresult')

        if self.is_equal(bact, 1): # BC
            if (self.is_missing(sputum) and self.is_missing(gene)) or \
               (self.is_in(sputum, [0, 5, 6]) and self.is_in(gene, [0, 1, 5, 6])):
                self.add_error(row, f"check_bactstatus: BC but results are negative/missing")
        elif self.is_equal(bact, 2): # CD
            if self.is_in(sputum, [1, 2, 3, 4]) or self.is_in(gene, [2, 3, 4, 7]):
                self.add_error(row, f"check_bactstatus: CD but found positive results")
        else:
            if self.is_in(sputum, [1, 2, 3, 4]) or self.is_in(gene, [2, 3, 4, 7]):
                self.add_error(row, f"check_bactstatus: Missing Bact_status but found positive results")


    def check_treatment_regime(self, row):
        tbcase = row.get('Case')
        reg = row.get('Treatment Registration')
        dsdrtb = row.get('Type of TB Treatment')
        regime = row.get('Treatment Regimens')
        age = row.get('Age_y')
        p_type = row.get('Type of Patient')

        age_val = age if not self.is_missing(age) else None

        if self.is_equal(tbcase, 1) and self.is_equal(reg, "Yes"):

            if age_val is not None:
                if age_val < 15:
                    if not self.is_missing(regime) and regime != "C":
                        self.add_error(row, f"check_regime: Child regime (C) expected for age {age_val}")
                else:
                    if not self.is_missing(regime) and regime == "C":
                        self.add_error(row, f"check_regime: Adult regime expected for age {age_val}")

            if not self.is_missing(regime):

                if self.is_equal(dsdrtb, "DS-TB"):
                    if self.is_in(regime, ["I", "C"]) and not self.is_in(p_type, ["N"]):
                        self.add_error(row, f"check_regime: Regime {regime} but patient {p_type} for DS-TB")

                    if self.is_in(regime, ["R"]) and self.is_in(p_type, ["N"]):
                        self.add_error(row, f"check_regime: Regime {regime} but patient {p_type} for DS-TB")

                    if self.is_in(regime, ["MDR"]):
                        self.add_error(row, f"check_regime: Regime {regime} for DS-TB")

                if self.is_equal(dsdrtb, "DR-TB"):
                    if self.is_in(regime, ["I", "R", "C"]):
                        self.add_error(row, f"check_regime: Regime {regime} for DR-TB")

        else:
            if self.is_equal(reg, "Y") and self.is_missing(regime):
                self.add_error(row, "check_regime: Treatment Registration is Yes but missing regime")

            if self.is_equal(reg, "N") and not self.is_missing(regime):
                self.add_error(row, f"check_regime: Treatment Registration is No but found regime {regime}")

            if self.is_equal(tbcase, 1) and (not self.is_equal(reg, "Y") or self.is_missing(regime)):
                self.add_error(row, f"check_regime: TB Case is 1 but Treatment Registration is {reg} and regime is {regime}")

    def check_completeness(self, row):
        tbcase = row.get('Case')
        reg = row.get('Treatment Registration')
        reason = row.get('Reasonforexamination')
        
        if self.is_missing(reason): 
            return
        else:
            reason = int(reason)
            if self.is_in(reason, [1, 2]):
                required = ['Date_x','MonthDiagnosis11', 'Team', 'Tsp', 'Approach', 'Visit_no', 'Sr_No', 'Name_x', 'Age_x', 'Case','VOL','Reasonforexamination']
            elif self.is_in(tbcase, [1, 2]):
                required += ['Month Diagnosis','Bacteriological status', 'Type of TB Treatment','StateRegion Name', 'Township Name','Bacteriological status','Volunteer Referral']
                if self.is_equal(reg, "Yes"): 
                    required += ['Township TB Reg Number', 'Type of Patient', 'TB Site', 'Treatment Regimens','Treatment Registrar Date', 'Initial Regimen Started Date','HIV Status','Facility Name Place of Registration', 'Reporting Period']

            missing_fields = [f for f in required if self.is_missing(row.get(f))]
            if missing_fields:
                self.add_error(row, f"check_completeness: Missing {', '.join(missing_fields)}")

    def check_sdgclinic(self, row):
        if self.is_missing(row.get('TypeofTBTreatment')):
            return
        else:
            sdg_epi = str(row.get('EPI11'))
            sdg_tb = str(row.get('TB11'))
            sr = int(row.get('Sr_No'))
            tsp = row.get('Tsp')
            approach = row.get('Approach')
            case = row.get('Case')
            if self.is_equal(tsp,"SDG") and self.is_equal(approach,"PPM"):
                if not self.is_in(sdg_epi, ["19","107","LDK"]):
                    self.add_error(row, f"check_sdgclinic: missing SDG_EPI for SDG PPM")
                    if self.is_equal(case, 1) and not self.is_in(sdg_tb, ["19","107","LDK"]):
                        self.add_error(row, f"check_sdgclinic: missing SDG_TB for TB case in SDG PPM")
                if 1 <= sr <= 1000 and not self.is_in(sdg_epi, ["19"]):
                    self.add_error(row, f"check_sdgclinic: SDG_EPI {sdg_epi} should be 19 for Sr_No {sr}")
                    if self.is_equal(case, 1) and not self.is_in(sdg_tb, ["19"]):
                        self.add_error(row, f"check_sdgclinic: SDG_TB {sdg_tb} should be 19 for TB case with Sr_No {sr}")
                if 1001 <= sr <= 2000 and not self.is_in(sdg_epi, ["107"]):
                    self.add_error(row, f"check_sdgclinic: SDG_EPI {sdg_epi} should be 107 for Sr_No {sr}")
                    if self.is_equal(case, 1) and not self.is_in(sdg_tb, ["107"]):
                        self.add_error(row, f"check_sdgclinic: SDG_TB {sdg_tb} should be 107 for TB case with Sr_No {sr}")
                if sr > 2000 and not self.is_in(sdg_epi, ["LDK"]):
                    self.add_error(row, f"check_sdgclinic: SDG_EPI {sdg_epi} should be LDK for Sr_No {sr}")
                    if self.is_equal(case, 1) and not self.is_in(sdg_tb, ["LDK"]):
                        self.add_error(row, f"check_sdgclinic: SDG_TB {sdg_tb} should be LDK for TB case with Sr_No {sr}")

    def check_case(self, row):
        if self.is_missing(row.get('Case')):
            return
        else:
            case = int(row.get('Case'))
            cxr = row.get('CXRresult211')
            bc = row.get('Bact_status')
            rfe = row.get('Reasonforexamination')
            age = row.get('Age_y')
            if self.is_equal(case, 1):
                if self.is_missing(bc):
                    self.add_error(row, f"check_case: Case is 1 but missing Bact_status")
                if self.is_missing(cxr):
                    self.add_error(row, f"check_case: Case is 1 but missing CXR result")
                if not self.is_in(cxr, [2,3,4]) and not self.is_equal(bc, 1):
                    self.add_error(row, f"check_case: Case is {case} but found Bact_status = {bc}, CXR result = {cxr}, Age = {age} and Reasonforexamination = {rfe}")
            else:
                if not self.is_missing(bc) and self.is_equal(bc, 1):
                    self.add_error(row, f"check_case: Case is {case} but found Bact_status = {bc}, Age = {age} and Reasonforexamination = {rfe}")
                if not self.is_missing(cxr) and self.is_equal(cxr, 2):
                    self.add_error(row, f"check_case: Case is {case} but found CXR result = {cxr}, Age = {age} and Reasonforexamination = {rfe}")
    
    def check_outcome(self, row):
        if self.is_missing(row.get('Treatment Outcome')):
            if self.is_missing(row.get('Treatment Outcome date')):
                return
            else:
                self.add_error(row, "check_outcome: Missing Treatment Outcome but found date")
        else:
            outcome = str(row.get('Treatment Outcome')).strip().lower()
            outcome_date = row.get('Treatment Outcome date')
            bact = row.get('Bacteriological status')
            if self.is_missing(outcome_date):
                self.add_error(row, f"check_outcome date: Missing Outcome date but found Outcome")
            if self.is_equal(outcome, "c"):
                if not self.is_equal(bact,"BC"):
                    self.add_error(row, f"check_outcome: Outcome {outcome} but not BC status")
    
    def check_volunteer(self, row):
        vol = row.get('VOL')
        pci = row.get('Referralfor')
        vol_name = row.get('VolunteerName')
        org = row.get('Organization')
        reason = row.get('Reasonforexamination')

        if self.is_equal(vol, 1):
            if self.is_missing(pci) or self.is_missing(vol_name) or self.is_missing(org):
                self.add_error(row, f"check_volunteerReferral: VOL is 1 but Referralfor {pci}, VolunteerName {vol_name}, or Organization {org}")
            if self.is_missing(reason) or not self.is_in(reason, [1, 2]):
                self.add_error(row, f"check_volunteerReferral: VOL is 1 but Reasonforexamination {reason}")
        else:
            if not self.is_missing(pci) or not self.is_missing(vol_name) or not self.is_missing(org):
                if not self.is_equal(vol, 1):
                    self.add_error(row, f"check_volunteerReferral: VOL is not 1 but found Referralfor {pci}, VolunteerName {vol_name}, or Organization {org}")

    def check_duplicate_tb_reg(self, row):
        tb_reg_num = row.get('TownshipTBRegNumber')
        township = row.get('Facility Name Place of Registration')
        facility = row.get('Placeforreferral')
        if self.is_missing(tb_reg_num):
            return
        composite_key = (tb_reg_num, township, facility)
        if composite_key in self.existing_tb_regs:
            self.add_error(
                row, 
                f"check_Duplicate TownshipTBRegNumber '{tb_reg_num}' "
                f"found for Township '{township}' and Facility '{facility}'"
            )
        else:
            self.existing_tb_regs.add(composite_key)

    def check_uncoding(self, row):
        if not self.is_equal(row.get('Case'), 1): return
        
        # Comparison logic using the mappings defined in __init__
        checks = [
            ('Sex_y', 'Sex_x', 'sex'),
            ('Treatment Registration', 'Treatmentreferral', 'yes_no'),
            ('Bacteriological status', 'Bact_status', 'bact_status'),
            ('Smoking Status', 'Smoking', 'smoking'),
            ('DM Status', 'DM1', 'dm'),
            ('Treatment Regimens', 'TreatmentRegimen', 'tx_regime'),
            ('Type of TB Treatment', 'TypeofTBTreatment', 'dsdr'),
            ('Xpert Result', 'GeneXpertresult', 'gene'),
            ('Facility Name Place of Registration', 'Placeforreferral', 'facility'),
            ('Xpert Result', 'GeneXpertresult', 'gene')  
        ]

        issues = []
        for t_col, p_col, map_key in checks:
            t_val = row.get(t_col)
            p_val = row.get(p_col)
            expected = self.mappings[map_key].get(p_val)
            if not self.is_missing(t_val) and not self.is_missing(p_val) and t_val != expected:
                issues.append(f"{map_key.capitalize()}: {p_val} vs {t_val}")

        if issues:
            self.add_error(row, f"check_uncoding: {', '.join(issues)}")

    def validate(self, df):
        """Main entry point to validate the dataframe."""
        for _, row in df.iterrows():
            self.check_date(row)
            self.check_patientid(row)
            self.check_CXR_GXP(row)
            self.check_bact_status(row)
            self.check_treatment_regime(row)
            self.check_completeness(row)
            self.check_sdgclinic(row)
            self.check_case(row)
            self.check_outcome(row)
            self.check_volunteer(row)
            self.check_uncoding(row)
            self.check_year_consistency(row)
            self.check_duplicate_tb_reg(row)

            # Add other specific checks like check_sdgclinic or check_formula here

        if not self.invalid_rows:
            return pd.DataFrame(columns=["InvalidData", "PatientID", "Approach", "Tsp"])
        
        return pd.DataFrame(self.invalid_rows)

In [8]:
def fuzzyMatchingData(dfmata):
    columns_to_check = ['Name_x', 'Age_x', 'Sex_x', 'Tsp', 'Reasonforexamination']
    required_columns = columns_to_check + ['PatientID', 'Address_x', 'Visit_no']
    missing_cols = [col for col in required_columns if col not in dfmata.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in input DataFrame: {missing_cols}")
    dfmata = dfmata[dfmata['Reasonforexamination']==1].copy()
    dfmata['DataCheck'] = dfmata[columns_to_check].astype(str).fillna('').agg('_'.join, axis=1)
    dfmata['Visit_no'] = pd.to_numeric(dfmata['Visit_no'], errors='coerce').fillna(0).astype(int)
    dataCheck = dfmata[['PatientID', 'DataCheck', 'Address_x', 'Visit_no']].copy()
    dataCheck_values = dataCheck['DataCheck'].tolist()
    scores_matrix = distance.cdist(
        [[val] for val in dataCheck_values],
        [[val] for val in dataCheck_values],
        lambda x, y: fuzz.ratio(x[0], y[0])
    )
    score_threshold = 85
    results = []
    for i in range(len(scores_matrix)):
        for j in range(i + 1, len(scores_matrix)):
            score = scores_matrix[i, j]
            if score >= score_threshold:
                visit1 = dataCheck['Visit_no'].iloc[i]
                visit2 = dataCheck['Visit_no'].iloc[j]
                duration = abs(visit2 - visit1)  # Ensure positive duration
                results.append({
                    'Score': score,
                    'ID': dataCheck['PatientID'].iloc[i],
                    'SimilarID': dataCheck['PatientID'].iloc[j],
                    'Patient': dataCheck['DataCheck'].iloc[i],
                    'SimilarPatient': dataCheck['DataCheck'].iloc[j],
                    'ScoreAddress': fuzz.ratio(str(dataCheck['Address_x'].iloc[i]), str(dataCheck['Address_x'].iloc[j])),
                    'Address': dataCheck['Address_x'].iloc[i],
                    'SimilarAddress': dataCheck['Address_x'].iloc[j],
                    'Visit1': visit1,
                    'Visit2': visit2,
                    'Duration': duration
                })
    df_fuzzy_duplicates = pd.DataFrame(results)
    if not df_fuzzy_duplicates.empty:
        df_fuzzy_duplicates.sort_values(by='Score', ascending=False, inplace=True)
        cols = ['Patient', 'SimilarPatient']
        split_values = df_fuzzy_duplicates[cols].astype(str).agg('_'.join, axis=1).str.split('_').tolist()

        dfnast = pd.DataFrame(split_values, columns=['Name', 'Age', 'Sex', 'Tsp', 'ReasonForExamination',
                                                     'Name_', 'Age_', 'Sex_', 'Tsp_', 'ReasonForExamination_'])
        comparison_cols = ['Name', 'Age', 'Sex', 'Tsp', 'ReasonForExamination']
        analysis = []
        for _, row in dfnast.iterrows():
            matched_attributes = [col for col in comparison_cols if row[col] == row[col + '_']]
            analysis.append({
                **{col: row[col] == row[col + '_'] for col in comparison_cols},
                'Analysis': matched_attributes
            })
        dfanalysis = pd.DataFrame(analysis)
        dfresult = pd.concat([df_fuzzy_duplicates.reset_index(drop=True), dfanalysis], axis=1)
    else:
        dfresult = pd.DataFrame(columns=[
            'Score', 'ID', 'SimilarID', 'Patient', 'SimilarPatient', 'ScoreAddress',
            'Address', 'SimilarAddress', 'Visit1', 'Visit2', 'Duration',
            'Name', 'Age', 'Sex', 'Tsp', 'ReasonForExamination', 'Analysis'
        ])
    return dfresult

In [9]:
def extract_features(name):
  name = name.lower()
  features = {
            'length': len(name),
            'vowel_count': len(re.findall('[aeiou]', name)),
            'has_repetition': 1 if len(set(name.split())) < len(name.split()) else 0,
            'first_two': name[:2] if len(name) >= 2 else name,
            'last_two': name[-2:] if len(name) >= 2 else name,
            'consonant_count': len(re.findall('[bcdfghjklmnpqrstvwxyz]', name)),
            'space_count': name.count(' ')
        }
  return features

class NameFeatureExtractor:
  def fit(self, X, y=None):
    return self

  def transform(self, X):
    features = [extract_features(name) for name in X]
    return np.array([[f['length'], f['vowel_count'], f['has_repetition'],
                            f['consonant_count'], f['space_count']] for f in features], dtype=float)

  def predict_gender(name, model=None, model_file='gender_predictor_model.pkl'):
    if model is None:
      model = joblib.load(model_file)
      prediction = model.predict([name])[0]
      probability = model.predict_proba([name])[0]
    return prediction, max(probability)

In [10]:
def filter_gender_results(df_input):
  df_input['confidence'] = pd.to_numeric(df_input['confidence'], errors='coerce')
  confidence_filter = df_input['confidence'] >= 0.5
  df_filtered = df_input[confidence_filter].copy()
  return df_filtered

In [11]:
def patientid_tpt(df):
    required_columns = {'Township', 'TPT Registration Number','Name'}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required column(s): {', '.join(missing_columns)}")
    df['PatientID'] = df['Township'].astype(str) + "_" + df['TPT Registration Number'].astype(str)+ "_" + df['Name'].astype(str)
    duplicate_ids = df['PatientID'][df.duplicated('PatientID', keep=False)]
    if not duplicate_ids.empty:
        raise ValueError(f"Duplicate IDs found: {duplicate_ids.tolist()}")
    return df

DATE_COLUMNS = ["Date","RegistrarDate","CPT","ART","Tx_Outcome_Date","DOTStartedDate111","TPTregistrationdate",
    "Drugpickup1","Date_drug_pickup2","Date_drug_pickup_3","Date_drug_pickup_4","Date_drug_pickup_5","Date_drug_pickup_6",
    "Date_drug_pickup_7","Date_drug_pickup_8","Date_drug_pickup_9","TPTdiscontinuationdate","OutcomeDate"]

def compare_data(dfCurrent: pd.DataFrame, dfPrevious: pd.DataFrame, id_col: str = "PatientID",date_cols: list = DATE_COLUMNS) -> pd.DataFrame:
    records = []
    dfC = dfCurrent.copy()
    dfP = dfPrevious.copy()
    shname = ", ".join(df2['SheetName'].astype(str).fillna("").unique())
    dfC[id_col] = dfC[id_col].astype(str).str.strip()
    dfP[id_col] = dfP[id_col].astype(str).str.strip()
    for col in date_cols:
        if col in dfC.columns:
            dfC[col] = pd.to_datetime(dfC[col], errors="coerce").dt.strftime("%Y-%m-%d")
        if col in dfP.columns:
            dfP[col] = pd.to_datetime(dfP[col], errors="coerce").dt.strftime("%Y-%m-%d")
    ids_dfCurrent = set(dfC[id_col].dropna().unique())
    ids_dfPrevious = set(dfP[id_col].dropna().unique())
    for pid in ids_dfCurrent - ids_dfPrevious:
        records.append([shname,pid, "_Not in Previous", pid, None])
    for pid in ids_dfPrevious - ids_dfCurrent:
        records.append([shname,pid, "_Not in Current", None, pid])
    cols_dfCurrent = set(dfC.columns) - {id_col}
    cols_dfPrevious = set(dfP.columns) - {id_col}
    for col in cols_dfCurrent - cols_dfPrevious:
        records.append([shname,col, "Missing Column in dfPrevious", "Exists", "Missing"])
    for col in cols_dfPrevious - cols_dfCurrent:
        records.append([shname,col, "Missing Column in dfCurrent", "Missing", "Exists"])
    common_cols = list((set(dfC.columns) & set(dfP.columns)) - {id_col})

    if common_cols:
        merged = pd.merge(
            dfC[[id_col] + common_cols],
            dfP[[id_col] + common_cols],
            on=id_col,
            how="inner",
            suffixes=("_Current", "_Previous"),
        )
        for col in common_cols:
            col_e = f"{col}_Current"
            col_s = f"{col}_Previous"
            for _, row in merged.iterrows():
                pid = row[id_col]
                norm1 = normalize_value(row[col_e])
                norm2 = normalize_value(row[col_s])
                if norm1 is None and norm2 is None:
                    continue
                if norm1 != norm2:
                    records.append([shname,pid, col, row[col_e], row[col_s]])
    return pd.DataFrame(records, columns=["SheetName","PatientID", "ColumnName", "Value_Current", "Value_Previous"])


def normalize_value(val):
    if pd.isna(val) or val is None or val is np.nan:
        return None
    s_val = str(val).strip()
    if s_val == "" or s_val.lower() in ("nan", "nat", "none", "null"):
        return None
    try:
        f_val = float(s_val)
        if f_val.is_integer():
            return str(int(f_val))
        return str(f_val)
    except (ValueError, TypeError):
        pass
    return s_val

### Checking Data Validation

In [12]:
ygntbpro_df = df_dict['YgnTBPro']
tb_df = df_dict['Notified TB Register']
df_combined = pd.merge(ygntbpro_df, tb_df, on="PatientID", how="outer")
if df_combined is not None and not df_combined.empty:
  df_combined.columns = [clean_column_name(col) for col in df_combined.columns]
  df_combined = changeDtype(df_combined)
  df_combined['EPI11'] = df_combined['EPI11'].astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', pd.NA)
  df_combined['TB11'] = df_combined['TB11'].astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', pd.NA)
  df_combined['SDG_EPI'] = df_combined['SDG_EPI'].astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', pd.NA)
  df_combined['SDG_TB'] = df_combined['SDG_TB'].astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', pd.NA)
  # dfValidation = validate_data(df_combined)
  validator = TBDataValidator()
  dfValidation = validator.validate(df_combined)
  dfValidation.sort_values(by='InvalidData', ascending=True, ignore_index=True, inplace=True)
  dfpresumptive = df_combined[df_combined['Reasonforexamination'].isin([1, 2])].copy()
  dfDuplication = fuzzyMatchingData(dfpresumptive)

['OLDTYPEOFTBTREATMENT', 'Initial Regimen Started Date']


KeyboardInterrupt: 

In [ ]:
import warnings
from sklearn.exceptions import InconsistentVersionWarning

warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

dfGender = df_combined[['PatientID', 'Name_x', 'Sex_x']].copy()
dfGender['Sex_x'] = dfGender['Sex_x'].astype(str)
dfGender['Sex_x']=dfGender['Sex_x'].replace({"1": 'M', "2": 'F'})
dfGender = dfGender.rename(columns={'Name_x': 'name', 'Sex_x': 'gender'})
dfGender['name'] = dfGender['name'].astype(str)
dfGender['gender'] = dfGender['gender'].astype(str)

try:
    model = joblib.load('gender_predictor_model.pkl')
    predictions = model.predict(dfGender['name'])
    probabilities = model.predict_proba(dfGender['name'])
    dfGender['predicted_gender'] = predictions
    dfGender['confidence'] = [max(prob) for prob in probabilities]
    checkGender = dfGender[dfGender['gender'] != dfGender['predicted_gender']]
    genderResult = checkGender.sort_values(by='confidence', ascending=False, inplace=False)

except FileNotFoundError:
    print("Error: 'gender_predictor_model.pkl' not found. Please ensure the model file exists or run the training code.")
    genderResult = pd.DataFrame(columns=['PatientID', 'name', 'gender', 'predicted_gender', 'confidence'])
except Exception as e:
    print(f"An error occurred during model loading or prediction: {e}")
    genderResult = pd.DataFrame(columns=['PatientID', 'name', 'gender', 'predicted_gender', 'confidence'])

dfGenderPredictor = filter_gender_results(genderResult)

In [ ]:
uniqueColumns = ['Team', 'StateRegionName', 'Tsp', 'WardVillage', 'Approach', 'Visit_no', 'ReportingPeriod',  'Nationality_x', 'Sex_x', 'Origin', 'VOL', 'Referralfor', 'Organization', 'Cough', 'Fever', 'Wtloss', 'Nightsweat', 'Haemoptysis', 'Chestpain', 'Fatigue', 'Neckglands', 'TBcontact', 'MDRTBcontact', 'TBTreatmenthistory', 'Transferin', 'TypeofPatients', 'TypeofDisease', 'SpecifyifEPSiteofEPTB', 'Smoking', 'Reasonforexamination', 'TypeofPatient', 'PublicHealthCare1', 'TypeofPatient1', 'DM1', 'RBS', 'HbA1c', 'HT1', 'Systolic', 'Diastolic', 'DMHT1', 'RTIAVI1', 'Generalweakness1', 'HIVStatus',  'Cxrr', 'CXRresult', 'Cxr_others', 'Sputum_request', 'Micror', 'Sputummicroscopyresult', 'Genexpertrequested', 'GeneXpertresult', 'XDRResult', 'Bact_status', 'Case', 'Treatmentreferral', 'TreatmentRegimen', 'Modifiedregimen', 'Placeforreferral', 'Ref1', 'CultureDSTresult', 'Microscope_Result_2nd_mnth', 'Xpert_Result_2nd_mnth', 'Microscope_Result_3rd_mnth', 'Xpert_Result_3rd_mnth', 'Microscope_Result_5th_mnth', 'Xpert_Result_5th_mnth', 'EndofTxMicroscopeResult', 'EndofTxXRayResult', 'EndofTxXpertResult', 'TreatmentOutcome1211', 'VolunteerGender111', 'VolunteerOrganization111', 'ContactInvestigation111', 'DOTSupervision111', 'DOTsupervisiontillTreatmentComp111', 'Seeing1', 'Hearing1', 'Walking1', 'Cognition1', 'Selfcare1', 'Communication1', 'Disability1', 'MonthDiagnosis11', 'Location11', 'EPI11', 'TB11', 'Xray2ndReading11', 'ReportingTsp11', 'CXRresult211', 'XrayEQA11', 'ReportingYear11', 'TPTregimen', 'DrugDose', 'TPTTreatmentOutcome', 'TypeofTBTreatment', 'SheetName_x', 'Month Diagnosis', 'Place of Notification', 'Ward', 'Volunteer Referral', 'Bacteriological status', 'Type of TB Treatment', 'Treatment Registration', 'StateRegion Name', 'Township Name', 'Facility Name Place of Registration', 'Reporting Period', 'Nationality_y', 'Sex_y', 'Refered from', 'Transfer in', 'Type of Patient', 'TB Site', 'Specify Site of EPTB if EP', 'Treatment Regimens', 'Smoking Status', 'DM Status', 'HIV Status', 'Microscope Result', 'X-Ray Result', 'Xpert Result', 'Culture Result', '2nd month Microscope Result', '2nd month Xpert Result', '3rd month Microscope Result', '3rd month Xpert Result', '5th month Microscope Result', '5th month Xpert Result', 'End of Tx Microscope Result', 'End of Tx X-Ray Result', 'End of Tx Xpert Result', 'Treatment Outcome', 'Volunteer Gender', 'Volunteer Organization', 'Contact Investigation', 'DOT  Supervision', 'DOT supervision till Treatment Completion', 'Seeing', 'Hearing', 'Walking', 'Cognition', 'Selfcare', 'Communication', 'Disability',  'SDG_EPI', 'SDG_TB', 'Location', 'Reporting Year', 'SheetName_y']
df_checkUnique = df_combined[uniqueColumns]

In [ ]:
def uniqueValue(df):
    unique_series = {}
    
    for col in df.columns:
        u_values = df[col].unique()       
        s = pd.Series(u_values).dropna()
        s = s[s.astype(str).str.strip() != '']
        s = s.sort_values().reset_index(drop=True)
        unique_series[col] = s
    return pd.concat(unique_series, axis=1)
df_unique = uniqueValue(df_checkUnique)
df_unique = df_unique.T
df_unique=df_unique.rename_axis('ColumnNames')
df_unique = df_unique.reset_index()
#df_unique

### Select Previous Report

In [ ]:
excelfilep = select_excel_file()
excelfilep

### Checking Difference

In [ ]:
if not excelfilep.value:
    raise FileNotFoundError("No file uploaded")
val = excelfilep.value
if isinstance(val, dict):
    file_data = next(iter(val.values()))
elif isinstance(val, tuple):
    file_data = val[0]
else:
    raise TypeError("Unknown FileUpload.value format")
file_bytes = file_data["content"]

def read_excel_bytes(file_bytes):
    required_sheets = ["YgnTBPro", "Presumptive Register", "Notified TB Register", "Outcome", "TPT Register"]

    with pd.ExcelFile(BytesIO(file_bytes), engine="openpyxl") as xf:
        try:
            dfs = pd.read_excel(
                        BytesIO(file_bytes),
                        sheet_name=required_sheets)
            for name in required_sheets:
               dfs[name]['SheetName'] = name
            return dfs
        except Exception:
            print("Available sheets:", xf.sheet_names)
            raise
df_dictp = read_excel_bytes(file_bytes)


In [ ]:
tpt_df = df_dict['TPT Register']
tpt_dfp = df_dictp['TPT Register']
#tpt_dfp = patientid_tpt(tpt_dfp)
#tpt_df = patientid_tpt(tpt_df)

df_list = ["ygntbpro_df","presumptive_df","tb_df","tpt_df","outcome_df"]
dfp_list = ["ygntbpro_dfp","presumptive_dfp","tb_dfp","tpt_dfp","outcome_dfp"]

columns = ["SheetName", "PatientID", "ColumnName", "Value_Current", "Value_Previous"]
df_diff = pd.DataFrame(columns=columns)

for key in df_dict.keys():
    df1 = df_dict[key].copy()
    shname1 = df1["SheetName"].unique().tolist()
    for key in df_dictp.keys():
      df2 = df_dictp[key].copy()
      shname2 = df2["SheetName"].unique().tolist()
      if shname1 == shname2:
        ############################################# changeDtype
        # if df1 is not None and not df1.empty:
        #   df1.columns = [clean_column_name(col) for col in df1.columns]
        #   # df1 = changeDftype(df1)
        #   df1 = changeDtype(df1)
        #   df1 = df1.astype(str)
        # if df2 is not None and not df2.empty:
        #   df2.columns = [clean_column_name(col) for col in df2.columns]
        #   # df2 = changeDftype(df2)
        #   df2 = changeDtype(df2)
        #   df2 = df2.astype(str)
        #############################################
        # df1 = df1.astype(str)
        # df2 = df2.astype(str)
        df = compare_data(df1, df2)
    if df is not None and not df.empty:
      df_diff = pd.concat([df_diff, df], axis=0, ignore_index=True)
df_diff = df_diff.replace({"nan": "", "NaT": "","<NA>":""})


### Downloading Data Verification Excel File

In [ ]:
dataframes = {
    'CheckData': globals().get('dfValidation'),
    'CheckGender': globals().get('dfGenderPredictor'),
    'CheckDuplication': globals().get('dfDuplication'),
    'CheckDifference': globals().get('df_diff'),
    'checkUnique': globals().get('df_unique')
}

timestamp_create = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
output_file_name = f"PythonDataVerification_{timestamp_create}"
file_path = fr"C:\Users\user\Downloads\vsoutput_{output_file_name}.xlsx"  ### WINDOWS
# file_path = fr"/Users/toepyaesone/Downloads/vsoutput_{output_file_name}.xlsx"  ### MAC
# file_path = fr"vsoutput_{output_file_name}.xlsx"  ### IOS

def is_valid_df(df):
    return isinstance(df, pd.DataFrame) and not df.empty

valid_dataframes = {
    sheet_name: df 
    for sheet_name, df in dataframes.items() 
    if is_valid_df(df)
}

if valid_dataframes:
    try:
        with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
            for sheet_name, df in valid_dataframes.items():
                df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"Successfully created Excel file at: {file_path}")
    except Exception as e:
        print(f"Error occurred while writing Excel file: {e}")
else:
    print("Warning: No valid or non-empty DataFrames found. Excel file was not created.")